# Create remaining NTASKS_OCN scaling cases

Runs each step as a fresh subprocess, streaming output live. This exists because going through
the `crocodash` MCP server's `create_case`/`process_forcings` tools blocks silently for a long
time with no visible progress, and re-running `Case()` repeatedly inside one long-lived process
(the MCP server) risks the same kind of global-state bleed seen in the CASEROOT singleton bug
for `configure_forcings`. Running each step as its own OS process avoids both: you get live
stdout, and each step is fully isolated.

**Updated plan (30-day runs, no per-case forcing copies):**
1. Reprocess the master case (`scaling_n1` / `scaling_test` inputdir) with a 30-day GLORYS
   window instead of the original 4-day one (`reprocess_master_30day.py`) — this is the slow
   GLORYS download+regrid step, run once.
2. Re-create `scaling_n2` (it currently has stale 4-day copied forcing files and hasn't been
   built/submitted yet, so nothing is lost) and create `scaling_n4/5/10/20`, all via
   `create_ntasks_case_workaround.py`. Each case gets its own fresh inputdir + uniquely-named
   grid (to avoid colliding with the master's entry in CIME's global grid registry), but
   instead of copying the master's processed forcing files, `user_nl_mom`'s `INPUTDIR` is
   repointed straight at the master's `ocnice/` folder — no file duplication needed.

Run cells one at a time, in order. If a cell seems stuck, interrupt just that cell (kernel
stays alive) and re-run it.

Kernel: `CrocoDash` conda env.

In [ ]:
import shutil
import subprocess
import sys
from pathlib import Path

CROC_CASES = Path("/glade/u/home/manishrv/croc_cases")
CROC_INPUT = Path("/glade/u/home/manishrv/scratch/croc_input")

def run_script(script_name: str, args=None):
    script = Path.cwd() / script_name
    proc = subprocess.Popen(
        [sys.executable, str(script)] + [str(a) for a in (args or [])],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in proc.stdout:
        print(line, end="", flush=True)
    ret = proc.wait()
    if ret != 0:
        raise RuntimeError(f"{script_name} {args or ''} exited {ret}")

def create_case(ntasks_ocn: int):
    run_script("create_ntasks_case_workaround.py", [ntasks_ocn])

## Step 1 — reprocess the master case for a 30-day GLORYS window

Slow (GLORYS download + regrid). Everything downstream (scaling_n1 itself, plus every
scaling_n{N} case's `user_nl_mom` INPUTDIR redirect) reads from `scaling_test/ocnice/`
afterward, so this must finish successfully before creating any per-NTASKS case.

In [ ]:
run_script("reprocess_master_30day.py")

## Step 2 — clear out the stale scaling_n2 (4-day, copy-based) attempt

Not built/submitted yet, so safe to remove and recreate with the new approach.

In [ ]:
for p in [CROC_CASES / "scaling_n2", CROC_INPUT / "scaling_n2"]:
    if p.exists():
        shutil.rmtree(p)
        print(f"Removed {p}")
    else:
        print(f"{p} does not exist, nothing to remove")

## Step 3 — create scaling_n2, n4, n5, n10, n20

In [ ]:
create_case(2)

In [ ]:
create_case(4)

In [ ]:
create_case(5)

In [ ]:
create_case(10)

In [ ]:
create_case(20)